# 08 - Merged Adapter Evaluation and Hyperparameter Search

This notebook performs two stages of analysis:

1. Evaluation of all merged LoRA adapters.
2. Hyperparameter search for selected merging methods.

The grid search explores:

- `alpha`;
- `top_k`;
- `drop_rate`.

For selected configurations, Pareto-optimal candidates are extracted.

In [4]:
import exp_common as E
import pandas as pd, json, itertools
from pathlib import Path

RUN_SEEDS = [42, 123, 777]
EVAL_LOGS_N = 2000
EVAL_QA_N = 500
BATCH_LOGS = 16
BATCH_QA = 4
SCENARIOS = ['ft_logs_plus_qa', 'cpt_plus_logs_plus_qa']
METHOD_DIRS = ['task_arithmetic', 'ties', 'model_soup', 'dare_plus_task_arithmetic', 'dare_plus_ties', 'dare_plus_model_soup']

In [ ]:
logs_eval = E.read_jsonl(E.PREPARED_DIR/'logs_test_eval.jsonl')[:EVAL_LOGS_N]
qa_eval = E.read_jsonl(E.PREPARED_DIR/'qa_test_eval.jsonl')[:EVAL_QA_N]
out_dir = E.RESULTS_DIR / 'merged'
out_dir.mkdir(parents=True, exist_ok=True)

def eval_adapter(label, adapter_dir, seed):
    print('Eval', label, seed, adapter_dir)
    lm = E.evaluate_logs(adapter_dir, logs_eval, out_dir/f'{label}_seed_{seed}_logs.json', batch_size=BATCH_LOGS)
    qm = E.evaluate_qa(adapter_dir, qa_eval, out_dir/f'{label}_seed_{seed}_qa.json', batch_size=BATCH_QA)
    return {'model': label, 'seed': seed, **{f'logs_{k}': v for k, v in lm.items() if not isinstance(v, list)}, **{f'qa_{k}': v for k, v in qm.items()}}

rows = []
for seed in RUN_SEEDS:
    for scenario in SCENARIOS:
        for method in METHOD_DIRS:
            adapter = E.MERGED_DIR / scenario / method / f'seed_{seed}'
            if (adapter/'adapter_model.safetensors').exists():
                rows.append(eval_adapter(f'{scenario}__{method}', adapter, seed))
            else:
                print('Missing, skip:', adapter)
    for seq in ['seq_logs_then_qa_then_cpt', 'seq_cpt_then_logs_then_qa', 'seq_cpt_then_qa_then_logs']:
        adapter = E.MERGED_DIR / seq / 'task_arithmetic' / f'seed_{seed}'
        if (adapter/'adapter_model.safetensors').exists():
            rows.append(eval_adapter(seq, adapter, seed))

df = pd.DataFrame(rows)
if (out_dir/'merged_metrics_raw.csv').exists():
    existing = pd.read_csv(out_dir/'merged_metrics_raw.csv')
    df = pd.concat([existing, df], ignore_index=True).drop_duplicates(subset=['model', 'seed'])
df.to_csv(out_dir/'merged_metrics_raw.csv', index=False)
df

In [ ]:
num_cols = [c for c in df.columns if c not in ['model', 'seed'] and pd.api.types.is_numeric_dtype(df[c])]
summary = df.groupby('model')[num_cols].agg(['mean', 'std']) if not df.empty else pd.DataFrame()
summary.to_csv(out_dir/'merged_metrics_mean_std.csv')
summary

In [ ]:
# Grid search is performed for one selected scenario
GRID_SEEDS = [123]
GRID_SCENARIO = 'cpt_plus_logs_plus_qa'
ALPHAS = [0.2, 0.4, 0.6, 0.8, 1.2]
TOP_KS = [0.1, 0.2, 0.3, 0.5]
DROP_RATES = [0.2, 0.4, 0.6]
RUN_GRID = True  # Enable this option to run grid search

grid_rows = []
if RUN_GRID:
    for seed in GRID_SEEDS:
        logs = E.ADAPTERS_DIR/'logs'/f'seed_{seed}'/'adapter'
        qa = E.ADAPTERS_DIR/'qa'/f'seed_{seed}'/'adapter'
        for alpha in ALPHAS:
            out = E.MERGED_DIR/'grid'/GRID_SCENARIO/f'ta_alpha_{alpha}'/f'seed_{seed}'
            E.merge_adapters([logs, qa], out, method='task_arithmetic', seed=seed, alpha=alpha)
            grid_rows.append(eval_adapter(f'grid_ta_alpha_{alpha}', out, seed))
        for top_k in TOP_KS:
            out = E.MERGED_DIR/'grid'/GRID_SCENARIO/f'ties_topk_{top_k}'/f'seed_{seed}'
            E.merge_adapters([logs, qa], out, method='ties', seed=seed, top_k=top_k)
            grid_rows.append(eval_adapter(f'grid_ties_topk_{top_k}', out, seed))
        for drop_rate in DROP_RATES:
            for method in ['dare+task_arithmetic', 'dare+ties', 'dare+model_soup']:
                safe = method.replace('+','_plus_')
                out = E.MERGED_DIR/'grid'/GRID_SCENARIO/f'{safe}_drop_{drop_rate}'/f'seed_{seed}'
                E.merge_adapters([logs, qa], out, method=method, seed=seed, drop_rate=drop_rate)
                grid_rows.append(eval_adapter(f'grid_{safe}_drop_{drop_rate}', out, seed))
    grid_df = pd.DataFrame(grid_rows)
    grid_df.to_csv(out_dir/'grid_metrics_raw.csv', index=False)
else:
    grid_df = pd.DataFrame()
grid_df

In [ ]:
# Pareto front based on two objectives: logs_f1_anomaly and qa_semantic_similarity_mean
pareto_source = grid_df if 'grid_df' in globals() and not grid_df.empty else df.copy()

def pareto_front(df, x='logs_f1_anomaly', y='qa_semantic_similarity_mean'):
    if df.empty or x not in df or y not in df:
        return pd.DataFrame()
    keep = []
    for i, row in df.iterrows():
        dominated = ((df[x] >= row[x]) & (df[y] >= row[y]) & ((df[x] > row[x]) | (df[y] > row[y]))).any()
        if not dominated:
            keep.append(i)
    return df.loc[keep].sort_values([x, y], ascending=False)

pareto = pareto_front(pareto_source)
pareto.to_csv(out_dir/'pareto_candidates.csv', index=False)
pareto

In [ ]:
# Grid search is performed for one selected scenario
GRID_SEEDS = [42]
GRID_SCENARIO = 'cpt_plus_logs_plus_qa'
ALPHAS = [0.2, 0.4, 0.6, 0.8, 1.2]
TOP_KS = [0.1, 0.2, 0.3, 0.5]
DROP_RATES = [0.2, 0.4, 0.6]
RUN_GRID = True  # Enable this option to run grid search

grid_rows = []
if RUN_GRID:
    for seed in GRID_SEEDS:
        logs = E.ADAPTERS_DIR/'logs'/f'seed_{seed}'/'adapter'
        qa = E.ADAPTERS_DIR/'qa'/f'seed_{seed}'/'adapter'
        for alpha in ALPHAS:
            out = E.MERGED_DIR/'grid'/GRID_SCENARIO/f'ta_alpha_{alpha}'/f'seed_{seed}'
            E.merge_adapters([logs, qa], out, method='task_arithmetic', seed=seed, alpha=alpha)
            grid_rows.append(eval_adapter(f'grid_ta_alpha_{alpha}', out, seed))
        for top_k in TOP_KS:
            out = E.MERGED_DIR/'grid'/GRID_SCENARIO/f'ties_topk_{top_k}'/f'seed_{seed}'
            E.merge_adapters([logs, qa], out, method='ties', seed=seed, top_k=top_k)
            grid_rows.append(eval_adapter(f'grid_ties_topk_{top_k}', out, seed))
        for drop_rate in DROP_RATES:
            for method in ['dare+task_arithmetic', 'dare+ties', 'dare+model_soup']:
                safe = method.replace('+','_plus_')
                out = E.MERGED_DIR/'grid'/GRID_SCENARIO/f'{safe}_drop_{drop_rate}'/f'seed_{seed}'
                E.merge_adapters([logs, qa], out, method=method, seed=seed, drop_rate=drop_rate)
                grid_rows.append(eval_adapter(f'grid_{safe}_drop_{drop_rate}', out, seed))
    grid_df = pd.DataFrame(grid_rows)
    grid_df.to_csv(out_dir/'grid_metrics_raw_42.csv', index=False)
else:
    grid_df = pd.DataFrame()
grid_df

In [ ]:
# Pareto front optimization: objective 1 - logs_f1_anomaly, objective 2 - qa_semantic_similarity_mean
pareto_source = grid_df if 'grid_df' in globals() and not grid_df.empty else df.copy()

def pareto_front(df, x='logs_f1_anomaly', y='qa_semantic_similarity_mean'):
    if df.empty or x not in df or y not in df:
        return pd.DataFrame()
    keep = []
    for i, row in df.iterrows():
        dominated = ((df[x] >= row[x]) & (df[y] >= row[y]) & ((df[x] > row[x]) | (df[y] > row[y]))).any()
        if not dominated:
            keep.append(i)
    return df.loc[keep].sort_values([x, y], ascending=False)

pareto = pareto_front(pareto_source)
pareto.to_csv(out_dir/'pareto_candidates_42.csv', index=False)
pareto